# Facility-Level Waste Allocation Pipeline

This notebook documents the pipeline that produces `data/processed/facility_waste_allocated.csv`.

## Overview

The pipeline allocates nationally-reported waste generation (from Eurostat `env_wasgen`) to individual E-PRTR industrial facilities using **pollutant emissions as allocation proxies**.

### Key Innovations

1. **Waste Type Validation**: Validates that allocated waste types match what each facility can actually produce based on its IED (Industrial Emissions Directive) activity classification.

2. **NACE Hierarchy Deduplication** (v2.0): Removes aggregate NACE codes when more detailed child codes exist, preventing double-counting. For example, if both 'C' (all manufacturing) and 'C24' (basic metals) exist for the same country/waste, only 'C24' is used.

3. **Recent Data Focus** (v2.0): Uses only the most recent N data points (default: 3 bi-annual periods = 6 years) instead of all historical data, ensuring allocations reflect current waste generation patterns.

### Data Flow

```
┌─────────────────────┐     ┌─────────────────────────┐
│  Eurostat env_wasgen │     │  E-PRTR Facility Data   │
│  (National waste by  │     │  (Facility emissions:   │
│   NACE × waste type) │     │   CO2, NOx, PM10)       │
└──────────┬──────────┘     └───────────┬─────────────┘
           │                             │
           │    ┌─────────────────────┐  │
           │    │  IED → NACE Mapping │  │
           │    │  IED → EWC Mapping  │  │
           │    └──────────┬──────────┘  │
           │               │             │
           ▼               ▼             ▼
     ┌─────────────────────────────────────────┐
     │       EmissionsAllocator                │
     │  1. NACE hierarchy deduplication        │
     │  2. Match facilities by country + NACE  │
     │  3. Validate waste types (IED → EWC)    │
     │  4. Calculate emission-weighted shares  │
     │  5. Allocate national waste             │
     └──────────────────┬──────────────────────┘
                        │
                        ▼
     ┌─────────────────────────────────────────┐
     │    facility_waste_allocated.csv         │
     │    (Facility-level allocations)         │
     └─────────────────────────────────────────┘
```

### Source Code Locations

| Component | File | Key Functions |
|-----------|------|---------------|
| Main allocator | `src/allocation/emissions_based_allocator.py` | `EmissionsAllocator`, `run_allocation_pipeline()` |
| E-PRTR loader | `src/loaders/eprtr.py` | `get_facility_emissions_summary()` |
| Eurostat loader | `src/loaders/eurostat.py` | `load_dataset()` |
| IED → NACE mapping | `src/mappings/ied_nace.py` | `IED_TO_NACE` dict |
| IED → EWC mapping | `src/mappings/ied_ewc_stat.py` | `is_waste_valid_for_ied()` |

---

## Data Quality Fixes (v2.0)

### Fix 1: NACE Double-Counting Prevention

**Problem**: Eurostat `env_wasgen` often contains both aggregate NACE codes (e.g., `'C'` for all manufacturing) and detailed codes (e.g., `'C24'` for basic metals). When both exist for the same country/waste combination, allocating from both causes double-counting.

**Solution**: The `_deduplicate_nace_hierarchy()` method removes parent codes when children are present:

```python
NACE_HIERARCHY = {
    'C': ['C10-C12', 'C13-C15', 'C16', 'C17_C18', 'C19',
          'C20-C22', 'C23', 'C24_C25', 'C26-C30', 'C31-C33'],
    'C24_C25': ['C24', 'C25'],
    # ... etc
}
```

A deduplication log is saved to `nace_dedup_log.csv` showing what was removed.

### Fix 2: Historical Data Bias Correction

**Problem**: The original loader calculated `mean_wasgen` over ALL available year columns (~2004-2024). This caused:
- Waste from 20 years ago allocated to facilities that may not have existed
- Obsolete waste streams inflated (e.g., Finnish wood waste W075 that's now recycled)
- Allocation doesn't reflect current patterns

**Solution**: New `n_datapoints` parameter (default=3) limits statistics to the most recent N data points. Since Eurostat waste data is bi-annual, 3 data points = 6 years.

Additionally, **standard error** is now calculated: `SE = std / sqrt(n)`:
- Series with 3 data points → lower SE (more confident)
- Series with only 2 data points → higher SE (less confident)
- Series with 1 data point → SE = NaN (no confidence)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent))

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

---

## Step 1: Load Waste Generation Data

National waste generation comes from Eurostat's `env_wasgen` dataset. This provides:
- Waste tonnage by country (`geo`)
- Economic sector (`nace_r2` - NACE Rev.2 codes)
- Waste type (`waste` - EWC-Stat codes like W061, W071, etc.)

**Key Parameters:**
- `n_datapoints` (default=3): Number of most recent bi-annual data points to use for calculating mean/std/se. Since data is bi-annual, 3 data points = 6 years.

**Output Columns:**
- `mean_wasgen`: Mean tonnes over the recent N data points
- `std_wasgen`: Standard deviation over recent data points
- `se_wasgen`: Standard error = std / sqrt(n_actual)
- `n_datapoints`: Number of actual non-null values used
- `years_used`: Which years were included in the calculation

**Source:** `src/loaders/eurostat.py:load_dataset()`

In [ ]:
from src.loaders.eurostat import load_dataset
import importlib
import src.loaders.eurostat
importlib.reload(src.loaders.eurostat)
from src.loaders.eurostat import load_dataset

# Load national waste generation data from Eurostat
# The load_dataset function:
#   - Fetches from Eurostat API using the 'eurostat' package
#   - Extracts tonnes-only rows (unit='T')
#   - Calculates mean/std/se across the most recent N data points (default: 3)
#   - Returns (dataset, labels, label_descriptions)

# n_datapoints=3 means use only the 3 most recent bi-annual data points (6 years)
# This prevents historical bias from 20-year-old waste patterns
wasgen, labels, label_desc = load_dataset('env_wasgen', n_datapoints=3)

print(f"Loaded {len(wasgen):,} records from env_wasgen")
print(f"\nColumns: {list(wasgen.columns)}")

# Show which years were used for statistics
if 'years_used' in wasgen.columns and len(wasgen) > 0:
    print(f"\nYears used for statistics: {wasgen['years_used'].iloc[0]}")

In [ ]:
# Examine the structure
print("Sample data:")
wasgen.head()

In [ ]:
# Key columns for allocation:
# - geo: country code (SE, NO, FI, DK, etc.)
# - nace_r2: NACE sector code (C24 = Basic metals, C25 = Fabricated metals, etc.)
# - waste: EWC-Stat waste classification (W061 = Ferrous metal waste, etc.)
# - mean_wasgen: average tonnes per year across the most recent N data points
# - se_wasgen: standard error = std / sqrt(n), for confidence indication

print(f"\nUnique countries: {wasgen['geo'].nunique()}")
print(f"Unique NACE codes: {wasgen['nace_r2'].nunique()}")
print(f"Unique waste types: {wasgen['waste'].nunique()}")

# Show the new statistics columns
if 'se_wasgen' in wasgen.columns:
    print(f"\nStatistics columns available:")
    print(f"  - mean_wasgen: ✓")
    print(f"  - std_wasgen: ✓")
    print(f"  - se_wasgen: ✓ (standard error for confidence)")
    print(f"  - n_datapoints: ✓ (actual data points used)")
    
    # Show example of standard error distribution
    print(f"\nStandard error distribution:")
    print(wasgen['se_wasgen'].describe())

---

## Step 2: Load Facility Emissions Data

E-PRTR (European Pollutant Release and Transfer Register) provides facility-level pollutant emissions used as allocation proxies.

**Key pollutants used:**
- **CO2** - Primary allocation proxy for energy-intensive industries
- **NOx** - Combustion indicator
- **PM10** - Particulate matter (process emissions indicator)

**Key Parameters:**
- `n_datapoints` (default=3): Number of most recent reporting years to include. Consistent with Eurostat loader to ensure both data sources use the same time window.

**Source:** `src/loaders/eprtr.py`

The loader:
1. Tries to load cached E-PRTR data from `data/raw/eprtr_facilities.csv`
2. Falls back to IED installations file (`F6_1_IED_Installations.csv`) if API unavailable
3. Filters emissions to recent reporting years (controlled by `n_datapoints`)
4. Generates synthetic emissions based on sector-specific emission factors if needed

In [ ]:
from src.loaders.eprtr import get_facility_emissions_summary

# Load facility data with emissions for Nordic countries
# This function (see src/loaders/eprtr.py:350):
#   1. Loads E-PRTR facilities from cache or IED file
#   2. Filters to most recent n_datapoints reporting years (consistent with Eurostat)
#   3. Pivots pollutant releases to get CO2, NOX, PM10 columns
#   4. Merges facility info (location, IED activity, etc.)

data_dir = Path('../data/raw')
nordic_countries = ['SE', 'NO', 'FI', 'DK']

# n_datapoints=3 ensures we use the same recent time window as Eurostat waste data
facilities = get_facility_emissions_summary(data_dir, countries=nordic_countries, n_datapoints=3)
print(f"Loaded {len(facilities):,} facilities")
print(f"\nColumns: {list(facilities.columns)}")

# Show which years were used for emissions data
if 'emissions_years_used' in facilities.columns:
    print(f"\nEmissions years used: {facilities['emissions_years_used'].iloc[0]}")

In [ ]:
# Examine facility data structure
# Key columns:
#   - facility_id: unique identifier
#   - ied_activity: IED Annex I activity code (e.g., '2.2' = Steel production)
#   - country_code: ISO 2-letter code
#   - lat, lon: geographic coordinates
#   - CO2, NOX, PM10: annual emissions (kg)

facilities[['facility_id', 'facility_name', 'country_code', 'ied_activity', 'CO2', 'NOX', 'PM10']].head(10)

In [ ]:
# Facility distribution by country and IED activity
print("Facilities by country:")
print(facilities['country_code'].value_counts())

print("\nTop 10 IED activities:")
print(facilities['ied_activity'].value_counts().head(10))

---

## Step 3: Understanding the Mappings

The allocation relies on two key mappings:

### 3.1 IED → NACE Mapping

Maps IED Annex I activity codes to NACE industry classification codes.

**Source:** `src/mappings/ied_nace.py`

In [ ]:
from src.mappings.ied_nace import IED_TO_NACE, get_nace_for_ied, get_ied_description

# Show some example IED → NACE mappings
example_ieds = ['2.2', '2.5(a)', '3.1(a)', '5.2']

print("Example IED → NACE mappings:")
print("-" * 70)
for ied in example_ieds:
    nace_codes = get_nace_for_ied(ied)
    desc = get_ied_description(ied)
    print(f"IED {ied}: {desc}")
    print(f"  → NACE: {nace_codes}")
    print()

### 3.2 IED → EWC-Stat Mapping (Waste Type Validation)

This is the **key innovation**: validates that a facility's IED activity can actually produce a given waste type.

For example:
- IED 2.2 (Steel production) CAN produce W061 (Ferrous metal waste) ✓
- IED 2.2 (Steel production) CANNOT produce W071 (Glass waste) ✗

**Source:** `src/mappings/ied_ewc_stat.py`

In [ ]:
from src.mappings.ied_ewc_stat import (
    is_waste_valid_for_ied, 
    get_waste_for_ied,
    get_primary_waste_for_ied
)

# Demonstrate waste type validation
print("Waste validation examples for IED 2.2 (Steel production):")
print("-" * 50)

test_wastes = ['W061', 'W062', 'W071', 'W124', 'W091']
for waste in test_wastes:
    is_valid = is_waste_valid_for_ied('2.2', waste)
    status = '✓ Valid' if is_valid else '✗ Invalid'
    print(f"  {waste}: {status}")

In [ ]:
# Show valid waste types for steel production
print("\nValid waste types for IED 2.2 (Steel production):")
print("-" * 50)

primary = get_primary_waste_for_ied('2.2', include_description=True)
print("Primary waste types:")
for w in primary:
    print(f"  {w['code']}: {w['description']}")

all_waste = get_waste_for_ied('2.2', include_description=True)
secondary = [w for w in all_waste if w not in primary]
print("\nSecondary waste types:")
for w in secondary:
    print(f"  {w['code']}: {w['description']}")

---

## Step 4: Sector Emission Weights

Different sectors have different emission profiles. The allocation uses sector-specific weights for combining CO2, NOx, and PM10 shares.

**Default weights:** CO2 = 55%, NOx = 30%, PM10 = 15%

**Source:** `data/processed/lookuptables/sector_emission_weights.csv`

In [ ]:
# Load sector emission weights
weights_path = Path('../data/processed/lookuptables/sector_emission_weights.csv')

if weights_path.exists():
    weights_df = pd.read_csv(weights_path)
    print("Sector emission weights:")
    print(weights_df)
else:
    print("Weights file not found - using defaults:")
    print("  CO2: 55%")
    print("  NOx: 30%")
    print("  PM10: 15%")

---

## Step 5: The Allocation Algorithm

The core allocation is performed by the `EmissionsAllocator` class.

**Algorithm for each (country, NACE, waste_type) combination:**

0. **NACE Hierarchy Deduplication (NEW):**
   - Check if parent NACE codes have children also present
   - Remove parent rows to prevent double-counting
   - Log removals to `nace_dedup_log.csv`

1. **Get national waste total:** `W_national = sum of all reported waste`

2. **Find matching facilities:**
   - Filter by country code
   - Filter by NACE code match (facility's IED → NACE → matches wasgen NACE)

3. **Validate waste type:**
   - Filter to only facilities whose IED activity can produce this waste type
   - Uses `is_waste_valid_for_ied()` function

4. **Calculate emission shares:**
   ```
   share_f = (w_CO2 × CO2_f/CO2_total) + 
             (w_NOx × NOx_f/NOx_total) + 
             (w_PM10 × PM10_f/PM10_total)
   ```

5. **Allocate waste:**
   ```
   W_facility = W_national × share_f
   ```

**Source:** `src/allocation/emissions_based_allocator.py:264-382` (calculate_emission_shares)

### NACE Hierarchy Deduplication Details

The deduplication uses a predefined hierarchy map:

```python
NACE_HIERARCHY = {
    'C': ['C10-C12', 'C13-C15', 'C16', 'C17_C18', 'C19',
          'C20-C22', 'C23', 'C24_C25', 'C26-C30', 'C31-C33'],
    'C24_C25': ['C24', 'C25'],
    # ...
}
```

For example, if data contains both `'C'` and `'C24'` for Finland/W061:
- `'C'` is the parent (all manufacturing)
- `'C24'` is the child (basic metals)
- The allocator removes the `'C'` row to avoid double-counting

In [ ]:
from src.allocation.emissions_based_allocator import EmissionsAllocator, load_emissions_allocator

# Create the allocator
# load_emissions_allocator() (see src/allocation/emissions_based_allocator.py:645):
#   1. Loads facility emissions via get_facility_emissions_summary()
#   2. Loads sector weights from lookuptable CSV
#   3. Returns initialized EmissionsAllocator with waste validation enabled

allocator = load_emissions_allocator(
    data_dir=data_dir,
    countries=nordic_countries,
    validate_waste_types=True  # Enable IED→EWC validation
)

print(f"Allocator initialized with {len(allocator.facilities)} facilities")
print(f"Waste type validation: {'ENABLED' if allocator.validate_waste_types else 'DISABLED'}")

In [ ]:
# Demonstrate the emission share calculation for a specific case
# This shows the intermediate calculation for Swedish steel (C24) producing ferrous metal waste (W061)

country = 'SE'
nace = 'C24'
waste = 'W061'

shares, status = allocator.calculate_emission_shares(country, nace, waste_code=waste)
print(f"Emission shares for {country}, {nace}, {waste}:")
print(f"Status: {status}")
print(f"Matched facilities: {len(shares)}")

if len(shares) > 0:
    print("\nFacility shares:")
    display_cols = ['facility_name', 'ied_activity', 'co2_share', 'nox_share', 'pm10_share', 'weighted_share']
    print(shares[display_cols].head(10))

---

## Step 6: Run the Full Allocation

Now we run the allocation for all waste streams.

**Source:** `src/allocation/emissions_based_allocator.py:418-538` (allocate_waste method)

In [ ]:
# Prepare wasgen data - rename columns to match allocator expectations
wasgen_input = wasgen.copy()
wasgen_input = wasgen_input.rename(columns={
    'geo': 'country',
    'nace_r2': 'nace',
    'mean_wasgen': 'tonnes'
})

# Filter to Nordic countries for this demonstration
wasgen_nordic = wasgen_input[wasgen_input['country'].isin(nordic_countries)]
print(f"Nordic waste streams to allocate: {len(wasgen_nordic):,}")
print(f"Total tonnage: {wasgen_nordic['tonnes'].sum():,.0f}")

In [ ]:
# Run the allocation
# allocate_waste() (src/allocation/emissions_based_allocator.py:438):
#   - Applies NACE hierarchy deduplication (if deduplicate_nace=True)
#   - Groups wasgen by (country, NACE, waste)
#   - For each group, calculates emission shares
#   - Validates waste types against IED activities
#   - Allocates national waste proportionally to facilities
#   - Tracks unallocated waste with reason codes

# deduplicate_nace=True removes parent NACE codes when children exist
allocated = allocator.allocate_waste(wasgen_nordic, deduplicate_nace=True)

print(f"\nAllocation results:")
print(f"  Total allocations: {len(allocated):,}")
print(f"  Tonnes allocated: {allocated['allocated_tonnes'].sum():,.0f}")
print(f"  Unique facilities: {allocated['facility_id'].nunique()}")

# Show NACE deduplication results if any
if hasattr(allocator, '_nace_dedup_log') and len(allocator._nace_dedup_log) > 0:
    print(f"\nNACE deduplication removed {len(allocator._nace_dedup_log)} duplicate rows")
    print("Sample deduplication log:")
    print(allocator._nace_dedup_log.head())

In [ ]:
# Show unallocated waste (if any)
if len(allocator.unallocated) > 0:
    print("\nUnallocated waste streams:")
    print(f"  Total streams: {len(allocator.unallocated)}")
    print(f"  Total tonnage: {allocator.unallocated['national_tonnes'].sum():,.0f}")
    
    print("\nBreakdown by reason:")
    reason_summary = allocator.unallocated.groupby('reason').agg({
        'national_tonnes': ['count', 'sum']
    })
    reason_summary.columns = ['n_streams', 'tonnes']
    print(reason_summary)
else:
    print("\nAll waste streams successfully allocated!")

In [ ]:
# Examine the output structure
print("Output columns:")
print(allocated.columns.tolist())

print("\nSample allocations (top 10 by tonnage):")
allocated.nlargest(10, 'allocated_tonnes')[[
    'facility_name', 'country', 'nace', 'waste_type', 
    'allocated_tonnes', 'weighted_share', 'method'
]]

---

## Step 7: Generate Allocation Summary

The `get_allocation_summary()` method provides coverage statistics by country and NACE sector.

**Source:** `src/allocation/emissions_based_allocator.py:585-642`

In [ ]:
# Generate summary statistics
summary = allocator.get_allocation_summary(allocated)

print("Allocation coverage summary:")
summary.head(15)

In [ ]:
# Overall coverage
total_allocated = summary['allocated_tonnes'].sum()
total_unallocated = summary['unallocated_tonnes'].sum()
overall_coverage = 100 * total_allocated / (total_allocated + total_unallocated)

print(f"\nOverall allocation coverage: {overall_coverage:.1f}%")
print(f"  Allocated: {total_allocated:,.0f} tonnes")
print(f"  Unallocated: {total_unallocated:,.0f} tonnes")

---

## Step 8: Running the Complete Pipeline

The `run_allocation_pipeline()` function wraps all steps and saves outputs.

**Source:** `src/allocation/emissions_based_allocator.py:792-888`

**Parameters:**
- `wasgen_path`: Path to waste data CSV or Eurostat code (e.g., 'env_wasgen')
- `output_dir`: Directory for output files
- `countries`: List of ISO country codes to process
- `validate_waste_types`: Enable IED→EWC validation (default: True)
- `deduplicate_nace`: Enable NACE hierarchy deduplication (default: True)
- `n_datapoints`: Number of recent data points for statistics (default: 3)

**Outputs produced:**
- `facility_waste_allocated.csv` - Main output with facility-level allocations
- `allocation_coverage_summary.csv` - Coverage statistics by country/NACE
- `unallocated_waste.csv` - Waste streams that couldn't be allocated (with reasons)
- `nace_dedup_log.csv` - Log of NACE rows removed to prevent double-counting (if any)

In [ ]:
from src.allocation.emissions_based_allocator import run_allocation_pipeline

# Run the complete pipeline with all quality fixes enabled
# This is equivalent to running from command line:
#   python -m src.allocation.emissions_based_allocator

output_dir = Path('../data/processed')

allocated_df, summary_df = run_allocation_pipeline(
    wasgen_path='env_wasgen',           # Load from Eurostat API
    output_dir=str(output_dir),
    countries=nordic_countries,
    validate_waste_types=True,          # Enable IED→EWC validation
    deduplicate_nace=True,              # Enable NACE hierarchy deduplication
    n_datapoints=3                      # Use only 3 most recent data points (6 years)
)

In [ ]:
# Verify output files
output_files = [
    'facility_waste_allocated.csv',
    'allocation_coverage_summary.csv',
    'unallocated_waste.csv',
    'nace_dedup_log.csv'  # New: NACE hierarchy deduplication log
]

print("Output files:")
for f in output_files:
    path = output_dir / f
    if path.exists():
        size = path.stat().st_size / 1024
        print(f"  ✓ {f} ({size:.1f} KB)")
    else:
        print(f"  ✗ {f} (not created - may be empty)")

---

## Output File Structure

### `facility_waste_allocated.csv`

| Column | Type | Description |
|--------|------|-------------|
| `facility_id` | str | Unique E-PRTR facility identifier |
| `facility_name` | str | Facility name |
| `country` | str | ISO 2-letter country code |
| `nace` | str | NACE sector code from waste data |
| `ied_activity` | str | IED Annex I activity code |
| `lat`, `lon` | float | Geographic coordinates |
| `waste_type` | str | EWC-Stat waste code |
| `allocated_tonnes` | float | Tonnes allocated to this facility |
| `allocated_se` | float | Standard error of allocated tonnes (confidence indicator) |
| `national_tonnes` | float | Total national waste for this stream |
| `national_se` | float | Standard error of national total |
| `co2_share` | float | Facility's share of CO2 emissions (0-1) |
| `nox_share` | float | Facility's share of NOx emissions (0-1) |
| `pm10_share` | float | Facility's share of PM10 emissions (0-1) |
| `weighted_share` | float | Final weighted allocation share (0-1) |
| `co2_weight_used` | float | CO2 weight applied (normalized) |
| `nox_weight_used` | float | NOx weight applied (normalized) |
| `pm10_weight_used` | float | PM10 weight applied (normalized) |
| `method` | str | Always 'emissions_weighted_validated' |

### Interpreting Standard Error

The `allocated_se` column indicates confidence in the allocation:
- **Lower SE** = more confident (consistent data across reporting periods)
- **Higher SE** = less confident (variable data or few data points)
- **NaN** = only 1 data point available (no SE calculable)

Use SE to identify allocations that may need verification or additional data sources.

In [ ]:
# Load and display the final output
final_output = pd.read_csv(output_dir / 'facility_waste_allocated.csv')

print(f"Final output: {len(final_output):,} facility-waste allocations")
print(f"Total allocated: {final_output['allocated_tonnes'].sum():,.0f} tonnes")

final_output.head()

---

## Appendix: Key Code References

For deeper exploration of the codebase:

### Main Allocator Class
- `src/allocation/emissions_based_allocator.py:38-55` - `NACE_HIERARCHY` constant for deduplication
- `src/allocation/emissions_based_allocator.py:58-113` - Class definition and initialization
- `src/allocation/emissions_based_allocator.py:264-400` - `calculate_emission_shares()` method
- `src/allocation/emissions_based_allocator.py:438-556` - `allocate_waste()` method with deduplication
- `src/allocation/emissions_based_allocator.py:605-675` - `_deduplicate_nace_hierarchy()` method
- `src/allocation/emissions_based_allocator.py:792-888` - `run_allocation_pipeline()` function

### Data Loaders
- `src/loaders/eprtr.py:350-415` - `get_facility_emissions_summary()` with n_datapoints
- `src/loaders/eprtr.py:178-270` - `_generate_emissions_from_sectors()` (synthetic emissions)
- `src/loaders/eurostat.py:7-65` - `load_dataset()` with n_datapoints and standard error

### Mappings
- `src/mappings/ied_nace.py:15-327` - `IED_TO_NACE` dictionary
- `src/mappings/ied_ewc_stat.py:177-199` - `is_waste_valid_for_ied()` function
- `src/mappings/ied_ewc_stat.py:29-54` - `_generate_ied_to_ewc_stat()` (chain: IED→BAT→EWC)

### New in v2.0
- `n_datapoints` parameter in both loaders to limit to recent data
- `se_wasgen` column for standard error confidence indication
- `deduplicate_nace` parameter to prevent double-counting
- `nace_dedup_log.csv` output file documenting NACE deduplication

In [ ]:
# Quick reference: show all IED activities and their descriptions
from src.mappings.ied_nace import IED_TO_NACE

ied_df = pd.DataFrame([
    {'ied_code': k, 'description': v['description'], 'nace_codes': ', '.join(v['nace'])}
    for k, v in IED_TO_NACE.items()
])

print("IED Annex I Activities:")
ied_df